#Excel出力

##インポート

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd

##ウィジェット・変数設定

In [0]:
dbutils.widgets.text("P_KIJYUN", "200504", "基準年月 (YYYYMM)")
p_kijyun = dbutils.widgets.get("P_KIJYUN")

yyyy = p_kijyun[:4]
mm = int(p_kijyun[4:6])

##データ読み込み

###マスタ

In [0]:
# 店舗マスタの読み込みと「店舗合計」行の追加
mise_mstr = spark.table("user_uenishi.mise_mstr").select("MISE_CD", "MISE_NM")
total_row = spark.createDataFrame([("9999", "店舗合計")], ["MISE_CD", "MISE_NM"])
full_mise_mstr = mise_mstr.union(total_row)

display(full_mise_mstr)

In [0]:
# 商品名称マスタ（大）とのデカルト積（全組み合わせ）
syohin_name_mstr = spark.table("user_uenishi.syohin_name_mstr1").select("SYOHIN_CD1", "SYOHIN_NM1")
base_mstr = full_mise_mstr.crossJoin(syohin_name_mstr)

display(base_mstr)

###売上データ

In [0]:
# 全月分の売上データを一括で読み込み
sales_list = []
for i in range(1, mm + 1):
    ym = f"{yyyy}{str(i).zfill(2)}"
    sales_list.append(spark.table(f"user_uenishi.conv_uri_{ym}"))

all_sales = sales_list[0]
for df in sales_list[1:]:
    all_sales = all_sales.union(df)

display(all_sales.limit(10))

##加工

In [0]:
# 価格マスタと結合し売上計算
kakaku_mstr = spark.table("user_uenishi.syohin_kakaku_mstr")
uriage_detail = all_sales.join(kakaku_mstr, "SYOHIN_CD", "inner") \
    .withColumn("URI_YM", F.date_format("YMD", "yyyyMM")) \
    .withColumn("SYOHIN_CD1", F.substring("SYOHIN_CD", 1, 2)) \
    .withColumn("URIAGE", F.col("KOSU") * F.col("SYOHIN_TANKA"))

# 店舗別の合計と「店舗合計(9999)」を算出して集計
# 1. 通常の店舗別集計
summary_mise = uriage_detail.groupBy("SYOHIN_CD1", "MISE_CD", "URI_YM").agg(F.sum("URIAGE").alias("URIAGE"))
# 2. 店舗合計(9999)の算出
summary_total = summary_mise.groupBy("SYOHIN_CD1", "URI_YM").agg(F.sum("URIAGE").alias("URIAGE")).withColumn("MISE_CD", F.lit("9999"))

display(summary_mise.limit(10))
display(summary_total.limit(10))

In [0]:
# 合流させて横持ち(Pivot)に変換
# 店舗CDと店舗名を組み合わせた列をPivot対象にする
summary_all = summary_mise.unionByName(summary_total, allowMissingColumns=True)
pivot_df = summary_all.groupBy("SYOHIN_CD1", "URI_YM").pivot("MISE_CD").sum("URIAGE")

display(pivot_df.limit(10))

In [0]:
# カテゴリごとの期間合計を算出して追加
window_cat = Window.partitionBy("SYOHIN_CD1")
final_df = pivot_df.withColumn("URI_YM", F.col("URI_YM").cast("string"))

# 数値列のみを対象に期間合計を計算
num_cols = [c for c in pivot_df.columns if c not in ["SYOHIN_CD1", "URI_YM"]]
total_sum_df = pivot_df.groupBy("SYOHIN_CD1").agg(*[F.sum(c).alias(c) for c in num_cols]).withColumn("URI_YM", F.lit("合計"))

# 最終データの合流
res_df = pivot_df.unionByName(total_sum_df).orderBy("SYOHIN_CD1", "URI_YM")

# データフレームの確認
display(res_df.limit(10))

##Excel出力

In [0]:
%pip install openpyxl -q

In [0]:
# 集計後のデータは比較的小さいため、Pandasに変換してExcel書き出し
import tempfile, shutil

output_path = f"/Volumes/training_bs/user_uenishi/output/uriage_jisseki_{p_kijyun}.xlsx"
pdf_all = res_df.toPandas()

# 商品カテゴリごとにシートを分けて保存
with tempfile.NamedTemporaryFile(suffix=".xlsx", delete=False) as tmp:
    tmp_path = tmp.name

with pd.ExcelWriter(tmp_path, engine='openpyxl') as writer:
    categories = pdf_all['SYOHIN_CD1'].unique()
    for cat in categories:
        pdf_cat = pdf_all[pdf_all['SYOHIN_CD1'] == cat]
        pdf_cat.to_excel(writer, sheet_name=cat, index=False)

shutil.copy(tmp_path, output_path)
import os; os.remove(tmp_path)

print(f"Excel出力完了: {output_path}")